In [10]:
from yarl import URL
from scripts.config import DATA_DIR, PRELOAD_FOLDER
from zipfile import ZipFile
import geopandas as gpd

import os

purefile = DATA_DIR / "pure" / "INEGI" / "UsoSueloMexico.zip"
rawfile = DATA_DIR / "raw" / "INEGI" / "UsoSueloMexico"

if not os.path.exists(rawfile):
    with ZipFile(purefile, "r") as mfile:
        mfile.extractall(rawfile)


suelo = gpd.read_file(rawfile / "conjunto_de_datos" / "cdv_usuev250sVII_cnal.shp")
suelo.shape

(179099, 3)

In [25]:
import re
classes = suelo['DESCRIPCIO'].unique().tolist()
agro_classes = [f for f in classes if re.search(r"AGRI\.*", f)]
agro = suelo[suelo['DESCRIPCIO'].isin(agro_classes)].reset_index(drop=True)
agro['Centro'] = agro['geometry'].centroid
agro

,CLAVE,DESCRIPCIO,geometry,Centro
0,HA,AGRICULTURA DE HUMEDAD ANUAL,"POLYGON ((3088803.469 423003.312, 3088771.741 ...",POINT (3082813.216 419667.208)
1,HA,AGRICULTURA DE HUMEDAD ANUAL,"POLYGON ((3194335.928 555090.026, 3194452.732 ...",POINT (3192866.277 554191.53)
2,HA,AGRICULTURA DE HUMEDAD ANUAL,"POLYGON ((3255422.905 596486.191, 3255394.188 ...",POINT (3254526.267 595878.215)
3,HA,AGRICULTURA DE HUMEDAD ANUAL,"POLYGON ((3255270.72 594680.826, 3254799.032 5...",POINT (3254959.289 596251.987)
4,HA,AGRICULTURA DE HUMEDAD ANUAL,"POLYGON ((3254657.643 599566.929, 3254483.622 ...",POINT (3253905.762 599246.072)
...,...,...,...,...
35662,TSP,AGRICULTURA DE TEMPORAL SEMIPERMANENTE Y PERMA...,"POLYGON ((2171382.466 1061442.055, 2171487.024...",POINT (2175908.219 1051759.589)
35663,TSP,AGRICULTURA DE TEMPORAL SEMIPERMANENTE Y PERMA...,"POLYGON ((2817250.362 1065407.968, 2817171.371...",POINT (2814912.871 1060204.295)
35664,TSP,AGRICULTURA DE TEMPORAL SEMIPERMANENTE Y PERMA...,"POLYGON ((2846715.249 1067029.961, 2846635.219...",POINT (2846707.683 1067901.768)
35665,TSP,AGRICULTURA DE TEMPORAL SEMIPERMANENTE Y PERMA...,"POLYGON ((2204435.926 1078602.259, 2204473.086...",POINT (2204664.111 1072435.774)


In [26]:
if not os.path.exists(DATA_DIR / "raw" / "INEGI" /"Mexico"):
    with ZipFile(PRELOAD_FOLDER / "mexico_agro.zip", "r") as mfile:
        mfile.extractall(DATA_DIR / "raw" / "INEGI" /"Mexico")

mexico_map = DATA_DIR / "raw" / "INEGI" /"Mexico"
mexico = gpd.read_file(mexico_map / "Mexico" /"conjunto_de_datos" / "00ent.shp")
mexico

,CVEGEO,CVE_ENT,NOMGEO,geometry
0,01,01,Aguascalientes,"POLYGON ((2469550.852 1159302.643, 2469738.209..."
1,02,02,Baja California,"MULTIPOLYGON (((1313480.513 1831458.607, 13135..."
2,03,03,Baja California Sur,"MULTIPOLYGON (((1694656.345 1227647.637, 16946..."
3,04,04,Campeche,"MULTIPOLYGON (((3544897.199 946994.621, 354491..."
4,05,05,Coahuila de Zaragoza,"POLYGON ((2469501.165 1978862.429, 2469539.651..."
5,06,06,Colima,"MULTIPOLYGON (((1157633.318 768573.525, 115772..."
6,07,07,Chiapas,"POLYGON ((3558179.782 698365.65, 3558487.689 6..."
7,08,08,Chihuahua,"POLYGON ((1961978.327 2201082.365, 1961978.967..."
8,09,09,Ciudad de México,"POLYGON ((2802175.66 843326.718, 2801806.564 8..."
9,10,10,Durango,"POLYGON ((2107050.444 1641943.398, 2109767.527..."


In [27]:
agroent = gpd.sjoin(mexico, agro, how="inner", predicate="contains").reset_index(drop=True)
agroent.drop(columns = ['index_right', 'CLAVE', 'CVEGEO'], inplace=True)

/tmp/ipykernel_45524/1135253261.py:1: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: EPSG:6372
Right CRS: PROJCS["Albers_Conic_Equal_Area",GEOGCS["GCS_GRS_1 ...

  agroent = gpd.sjoin(mexico, agro, how="inner", predicate="contains").reset_index(drop=True)


In [28]:
agroent['Centro']=agroent['Centro'].to_crs(epsg=4326)
agroent

,CVE_ENT,NOMGEO,geometry,DESCRIPCIO,Centro
0,01,Aguascalientes,"POLYGON ((2469550.852 1159302.643, 2469738.209...",AGRICULTURA DE RIEGO ANUAL Y SEMIPERMANENTE,POINT (-102.28592 21.72969)
1,01,Aguascalientes,"POLYGON ((2469550.852 1159302.643, 2469738.209...",AGRICULTURA DE TEMPORAL ANUAL,POINT (-102.40975 21.77244)
2,01,Aguascalientes,"POLYGON ((2469550.852 1159302.643, 2469738.209...",AGRICULTURA DE TEMPORAL ANUAL,POINT (-102.5158 21.79584)
3,01,Aguascalientes,"POLYGON ((2469550.852 1159302.643, 2469738.209...",AGRICULTURA DE RIEGO ANUAL,POINT (-102.46886 21.81088)
4,01,Aguascalientes,"POLYGON ((2469550.852 1159302.643, 2469738.209...",AGRICULTURA DE TEMPORAL ANUAL,POINT (-102.5426 21.81567)
...,...,...,...,...,...
33670,32,Zacatecas,"POLYGON ((2470352.986 1453426.983, 2473254.843...",AGRICULTURA DE TEMPORAL ANUAL,POINT (-101.70467 24.75622)
33671,32,Zacatecas,"POLYGON ((2470352.986 1453426.983, 2473254.843...",AGRICULTURA DE TEMPORAL ANUAL,POINT (-101.7887 24.86394)
33672,32,Zacatecas,"POLYGON ((2470352.986 1453426.983, 2473254.843...",AGRICULTURA DE TEMPORAL ANUAL,POINT (-102.12415 24.78334)
33673,32,Zacatecas,"POLYGON ((2470352.986 1453426.983, 2473254.843...",AGRICULTURA DE TEMPORAL ANUAL,POINT (-102.13454 24.79761)


In [30]:
agropoints = agroent['Centro'].to_list()
agrocoords = [p.coords[0] for p in agropoints]

agrocoords

[(-102.28591602335558, 21.72969343327807),
 (-102.40975006811368, 21.77243531020748),
 (-102.51579591803714, 21.795837141742023),
 (-102.4688593743015, 21.810878688139404),
 (-102.54260394243583, 21.815668400078007),
 (-102.47401208108364, 21.82111352485242),
 (-102.50706015594378, 21.824792441987075),
 (-102.5353547274295, 21.850170059294136),
 (-102.30092512226852, 21.744920018325043),
 (-102.36349321269195, 21.750914530967385),
 (-102.3405643280743, 21.766569080950305),
 (-102.32019671495242, 21.77945372415422),
 (-102.30410332172106, 21.7850455482622),
 (-102.30532013432367, 21.79488474179825),
 (-102.3366257919657, 21.799175103435783),
 (-102.26830043879052, 21.802961308395794),
 (-102.27065937253349, 21.814009134288003),
 (-102.27052003010203, 21.837922361046775),
 (-102.78681711569446, 21.823016866206785),
 (-102.12012915878556, 21.83274016112649),
 (-102.11251584526886, 21.868210991013513),
 (-102.07252865853124, 21.93334212423573),
 (-102.4542460534553, 22.162783053927793),
 (

In [34]:
agroent.geometry = agroent['Centro']

In [36]:
# Tu lista de datos


# Abrir (o crear) el archivo en modo escritura ('w')
with open("agrocoords.txt", "w", encoding="utf-8") as archivo:
    for elemento in agrocoords:
        archivo.write(f"{elemento}\n")

